# EuroLeague Fantasy: weekly picks

Run this before each round's deadline (the first tip-off). A GitHub Action saves the app's
line-ups at 14:41 UTC on game days; running after that uses the freshest injury news.

1. Adjust the settings cell (budget, players you know are out, optional roster).
2. **Run all**. It refreshes the data, trains the models on all history (1-2 min) and
   picks the best 11 within your budget.
3. With `SAVE = True` the result is saved under `recommendations/` (used for tracking).

Line-ups are fetched live from the app on every run, so run it as late as possible
before the deadline (the first tip-off). It needs an internet connection.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from elfantasy.weekly import recommend

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

## Settings

In [ ]:
MATCHDAY = None        # None = the next round; or a round number, e.g. 15
BUDGET = 100.0         # your team value + bank if it has grown
REFRESH = True         # fetch new box scores and the current player list first

# Injury news the app has not picked up yet: 'OUT' (0%), 'DOUBTFUL' (30%),
# 'PROBABLE' (75%) or 'FIT' (use the model even if the app lists him as out)
OVERRIDES = {
    # 'Mike James': 'OUT',
}
LOCKED = []            # players that must be in the squad
EXCLUDED = []          # players to leave out

# Optional: your current 10 players + coach (names) to get the best trades instead
ROSTER = None          # e.g. ['Sasha Vezenkov', ..., 'Georgios Bartzokas']
BANK = 0.0             # unspent credits
MAX_TRADES = 4         # 4 per round; raise it during an unlimited window
SAVE = True           # save the result to recommendations/ (set False for test runs)

In [ ]:
rec = recommend(matchday=MATCHDAY, budget=BUDGET, overrides=OVERRIDES, locked=LOCKED,
                excluded=EXCLUDED, roster=ROSTER, bank=BANK, max_trades=MAX_TRADES,
                refresh=REFRESH)
print(f'Round {rec.matchday} | deadline {rec.deadline:%a %d %b %H:%M} UTC | '
      f'expected {rec.squad.objective:.1f} pts | cost {rec.squad.cost:.1f} / {rec.budget:.1f}')
for n in rec.notes:
    print('note:', n)

## Best 11

`exp_pts` = P(plays) × points if he plays. Captain marked (C).

In [ ]:
display(rec.squad_table())
rec.coach_pick()

## Captain shortlist

In [ ]:
rec.captain_shortlist()

## Rankings by position

In [ ]:
for pos in ('G', 'F', 'C'):
    display(Markdown(f'### {pos}'))
    display(rec.rankings(pos, n=10))

## Value picks (expected points per credit)

In [ ]:
rec.value_picks(n=12)

## Coaches

In [ ]:
rec.coach_table()

## Why these picks

Each feature's contribution (fantasy points) to the prediction, relative to the average player.

In [ ]:
for name in rec.squad_table()['name']:
    display(Markdown(f'**{name}**'))
    display(rec.why(name).head(5))

## Trades for your roster (if `ROSTER` is set)

In [ ]:
if rec.trades:
    print('Out:', ', '.join(rec.trades['out']) or 'none')
    print('In: ', ', '.join(rec.trades['in']) or 'none')
    print(f"Expected gain: {rec.trades['gain']:+.1f} pts")
    display(Markdown('**Your squad after these trades**'))
    display(rec.my_squad_table())
    display(rec.coach_pick(rec.my_squad))
else:
    print('No roster given.')

## Save

In [ ]:
if SAVE:
    print('saved to', rec.save())
else:
    print('not saved (SAVE = False)')